# Calibrating the Bakshi-Cao-Chen Framework to Equity Derivatives
## Jump Risk Estimation and Volatility Dynamics in the South African Market Context

---

#### Author: Lefaso Azael Machere
#### Programme: Certificate in Python for Finance (CPF) — The Python Quants
#### Date: June 2026

## Abstract

The Black-Scholes-Merton (BSM) model remains the industry
standard for equity option pricing, yet it systematically
fails to capture two features observed in real markets:
sudden discontinuous price jumps and volatility that changes
randomly over time. These limitations are particularly acute
in South Africa, where political events — including the
Nenegate cabinet reshuffle (December 2015), the Zuma recall
(February 2018), and the COVID-19 market collapse (March 2020)
— have produced sharp sudden index drops that smooth diffusion
models cannot represent.

This project implements and validates the Bakshi-Cao-Chen (BCC)
framework — combining Heston (1993) stochastic volatility,
Merton (1976) jump-diffusion, and Cox-Ingersoll-Ross (1985)
stochastic interest rates — as a more realistic alternative.
The calibration pipeline is validated on Euro Stoxx 50 option
data, establishing methodological rigour against known
benchmarks. SA-specific jump parameters are then estimated
from JSE Top40 historical daily returns using jump-filtering
techniques, isolating the discontinuous return component that
BSM ignores entirely.

Results demonstrate that the BCC model produces materially
lower calibration error than BSM, Merton, or Heston alone —
consistent with the finding of Galluccio and Le Cam (2008)
that simultaneous jumps and stochastic volatility are required
to explain the observed volatility smile. Estimated SA jump
parameters differ materially from European calibrations,
reflecting the concentrated political risk that characterises
the South African equity market. These findings have direct
implications for derivatives pricing and risk management at
South African investment banks.

## 1. Environment Setup

We begin by importing the Python libraries required throughout
this project. Each library serves a specific role in the
analysis pipeline:

- **numpy** and **pandas** handle numerical arrays and
  tabular data respectively
- **scipy** provides the optimisation routines for model
  calibration and numerical integration for the Lewis
  transform pricing formula
- **matplotlib** produces all visualisations

We also configure consistent display settings for figure
size, grid lines, and print precision so all outputs are
readable and reproducible across environments.

In [1]:
# ── Standard library ──────────────────────────────────────────
import math                          # elementary math functions
import json                          # loading pre-computed parameters
from pathlib import Path             # file path handling

# ── Numerical and data ────────────────────────────────────────
import numpy as np                   # numerical arrays and operations
import pandas as pd                  # tabular data handling

# ── Optimisation and integration ──────────────────────────────
from scipy.integrate import quad     # numerical integration (Lewis transform)
from scipy.optimize import fmin, brute  # local and global optimisation

# ── Plotting ──────────────────────────────────────────────────
import matplotlib.pyplot as plt      # visualisation
import matplotlib as mpl             # figure configuration

# ── Display settings ──────────────────────────────────────────
np.set_printoptions(precision=6, suppress=True)
mpl.rcParams['figure.figsize'] = (10.0, 6.0)
mpl.rcParams['axes.grid'] = True
plt.style.use('seaborn-v0_8')

print("Environment ready.")
print(f"NumPy:   {np.__version__}")
print(f"pandas:  {pd.__version__}")

Environment ready.
NumPy:   2.0.2
pandas:  2.2.2


## 2. Introduction

The Black-Scholes-Merton (BSM) model, introduced in 1973,
transformed the derivatives industry by providing the first
closed-form option pricing formula. Despite its theoretical
elegance, BSM rests on assumptions that are systematically
violated in real markets. Most critically, it assumes that
asset prices follow a continuous diffusion process with
constant volatility. In practice, equity markets exhibit
sudden discontinuous price jumps, volatility that clusters
and mean-reverts over time, and interest rates that are
themselves stochastic. These violations produce the
well-documented volatility smile — a pattern of implied
volatilities across strikes that BSM predicts should be
flat but that every liquid options market shows is curved.

South African equity markets amplify these failures in a
distinctive way. The JSE Top40 index has experienced
several sharp, politically-driven discontinuous moves
that no diffusion model can reproduce. The Nenegate
cabinet reshuffle of December 2015 — in which President
Zuma unexpectedly replaced the finance minister — caused
the South African rand to lose approximately 10% of its
value overnight, with severe knock-on effects on equity
markets. The recall of President Zuma in February 2018
and the COVID-19 market collapse of March 2020, during
which the JSE Top40 fell approximately 35% peak to
trough, produced similarly abrupt index movements. These
events are not outliers to be dismissed — they are
structural features of a market exposed to concentrated
political risk, and any pricing model applied to SA
derivatives must account for them.

The Bakshi, Cao and Chen (1997) framework — hereafter
BCC — was designed precisely to address these limitations.
By combining Heston (1993) stochastic volatility, Merton
(1976) jump-diffusion, and Cox-Ingersoll-Ross (1985)
stochastic short rates into a single unified model, BCC
can simultaneously capture the volatility smile, the
leverage effect, sudden price jumps, and a realistic
interest rate term structure. Galluccio and Le Cam (2008)
demonstrated that no single-feature model — whether
stochastic volatility alone or jumps alone — can
adequately reproduce the observed smile; both features
are required simultaneously. While BCC has been
extensively studied in European and US markets, no
published work has examined how its parameters —
particularly the jump component — manifest in the
South African market context.

This project addresses that gap. Specifically, it
pursues three objectives: first, to implement and
validate the full BCC calibration pipeline against
the Euro Stoxx 50 benchmark dataset; second, to
calibrate the CIR short-rate model to the South
African yield curve constructed from market data;
and third, to estimate SA-specific jump parameters
from JSE Top40 historical returns using jump-filtering
techniques and compare these to European calibrations.
The project tests three hypotheses: that BCC produces
materially lower calibration error than nested
sub-models; that SA jump parameters differ materially
from European benchmarks; and that BSM systematically
underprices options in the SA context relative to BCC.